# This code process the output file from fragpipe when spectra is mathced with merged.fasta.transdecoder210506.pep.fasta


In [33]:
#Import pandas for data manipulation
import pandas as pd
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)   # prevents line wrapping

#Define sample set name and ID
sample_set = "AE2520Fadime"
first_sample_number = 1 #this is to match sample number in extraction with sample name. Only works if numbers are consecutive and do not skip any numbers in between


# Input and output files
prot_file_fragpipe = "../3_data/3_metaP/AE2520_Clio_57to60_Arianna260724db_final_report.pg_matrix_annotation_report_full.tsv" # add name of metaproteromics file from fragpipe
modified_prot = f"modified_report.pg_matrix_{sample_set}.csv" # add name of cleaned metaproteomics file
#metadata file name

#Read metadata file with original names
meta_names = pd.read_csv(f"../data/metaP/Metadata_{sample_set}.csv")
meta_names.columns = meta_names.columns.str.replace(" ", "_") # Removes spaces from names

## Open metaproteomics file and clean it up so it matches annotations file

In [34]:
# Keep only the read's name in the Protein.Group column to match perfectly with the annotations. Clears everything after the first .p[1-9]
# Read CSV (skip "#" in header if present)
p = pd.read_csv(prot_file_fragpipe, comment="#")
# Modify first column: truncate everything after the first `.p###`
p.iloc[:, 0] = p.iloc[:, 0].str.replace(r'(\.p\d+).*', r'\1', regex=True)
#drop unnecessary columns
p = p.drop(p.columns[[1,2,3,4]], axis=1)  # Drops columns except protein ID
# Set names of rows from modified proteomic data set
p.columns = p.columns.str.replace(" ", "_") # Removes spaces from names
# Uses attribute index from Panda to use column "Protein.Group"  as row names
p.index = p["Protein.Group"]
#Remove the "Protein.Group" column as an actual column. The Protein.Group names will still remain as the index in the dataframe
p = p.drop(columns=["Protein.Group"])
#Replace NaN values with 0
p = p.fillna(0)
display(p)

,3_UW5_02,43_UW91_02,24_UW73_51,31_UW114_02,18_UW109_51,63_UW104_02,6_UW9_02,37_UW81_02,53_UW98_02,56_UW100_51,41_UW89_02,35_UW77_02,49_UW96_02,33_UW77_51,24_UW73_51.1,19_UW70_02,61_UW103_02,13_UW28_02,55_UW99_02,38_UW85_51,40_UW89_51,60_UW103_51,54_UW99_51,48_UW96_51
Protein.Group,,,,,,,,,,,,,,,,,,,,,,,,
megahit_HN001_k141_100230.p1,2660000.0,2110000.0,0.0,0.0,0.0,0.0,2160000.0,3900000.0,3320000.0,0.0,3036200.0,2895490.0,3451240.0,577764.0,958444.0,3242860.0,677613.0,2713120.0,858822.0,934814.0,1177800.0,0.0,0.0,620485.0
megahit_HN001_k141_100427.p1,1560000.0,1860000.0,0.0,673808.0,1100000.0,1270000.0,1390000.0,2270000.0,1750000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
megahit_HN001_k141_100444.p1,1400000.0,2580000.0,0.0,1680000.0,193037.0,1800000.0,1790000.0,2250000.0,1700000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
megahit_HN001_k141_100576.p1,2410000.0,4490000.0,194628.0,718187.0,325810.0,765412.0,1350000.0,3680000.0,4760000.0,466451.0,4094560.0,4724080.0,7396130.0,369465.0,0.0,2772890.0,907656.0,1987500.0,1324480.0,587246.0,1102220.0,0.0,353534.0,0.0
megahit_HN001_k141_100648.p1,521980.0,1620000.0,0.0,2310000.0,0.0,1230000.0,568076.0,1830000.0,1820000.0,0.0,1232420.0,1089370.0,1610180.0,682861.0,0.0,674306.0,1099260.0,882058.0,2367410.0,576764.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
velvet_HN044_NODE_521783_length_270_cov_4.903704.p1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,154108.0,0.0,684507.0,0.0,0.0,0.0,0.0,0.0,473245.0,165950.0,0.0,0.0,0.0
velvet_HN044_NODE_546770_length_555_cov_4.526126.p1,0.0,0.0,1500000.0,0.0,4490000.0,0.0,295854.0,472801.0,0.0,3330000.0,0.0,0.0,241415.0,1769750.0,1223920.0,0.0,0.0,563779.0,0.0,1904240.0,558223.0,15273100.0,7237200.0,908701.0
velvet_HN044_NODE_551706_length_275_cov_6.549091.p1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,148365.0,196716.0,0.0,187022.0,0.0,80401.1,0.0,162956.0,0.0,617142.0,0.0,0.0,0.0,0.0


## Extract sample names from FragPipe and merge with original sample ID's from metadata

In [35]:
# Extract column headers (sample names) into new DataFrame to merge with original sample names and add them to the proteomics data
# Extract column headers
samples_id = pd.DataFrame({"Sample_ID": p.columns})

# Extract UW number (digits after 'UW')
samples_id["UW_num"] = (
    samples_id["Sample_ID"]
    .str.extract(r"UW(\d+)")
    .astype(int)
)

# Sort by UW number
samp_names = samples_id.sort_values(by="UW_num")

# Add increasing IDs
samp_names["ID"] = range(
    first_sample_number,
    first_sample_number + len(samp_names)
)

# Optional: drop helper column
samp_names = samp_names.drop(columns="UW_num")

display(samp_names)

,Sample_ID,ID
0,3_UW5_02,1
6,6_UW9_02,2
17,13_UW28_02,3
15,19_UW70_02,4
2,24_UW73_51,5
14,24_UW73_51.1,6
13,33_UW77_51,7
11,35_UW77_02,8
7,37_UW81_02,9
19,38_UW85_51,10


## Open taxonomic file and clean

In [36]:
# Import taxonomic dataset: newtaxa.eukprot.merged.fasta.transdecoder-estimated-taxonomy.out
tax = pd.read_csv("../data/metaP/transcriptome_db/newtaxa.eukprot.merged.fasta.transdecoder-estimated-taxonomy.out", sep="\t")

#Make a variable "aa" to clean data
aa = tax.sort_values(by="max_pid") #sort DataFrame by max_pid column
aa = aa.drop(columns=[col for col in aa.columns if col == "Unnamed: 0"]) # drop column with empty data from file
bb = aa.drop_duplicates(subset="transcript_name") #Remove duplicate row based on "transcript_name", keeping the firts occurance

#Assign the cleaned DataFrame back to "tax"
tax = bb.copy()
#Set row names (index) to 'transcript_name'
tax.index = tax["transcript_name"]
tax = tax.drop(columns=["transcript_name"])

# Split 'full_classification' column into multiple columns using ';' as a delimiter
# Define expected column names
classification_columns = ["domain", "supergroup", "division", "class", "order", "family", "genus", "species"]

# Split, strip spaces, reindex, and fill missing values with "Unknown"
tax[classification_columns] = (
    tax["full_classification"]
    .str.split(";", expand=True)                 # split into multiple columns
    .apply(lambda col: col.str.strip())         # remove spaces
    .reindex(columns=range(len(classification_columns)))  # ensure 8 columns
    .fillna("Unknown")                          # replace NaN with "Unknown"
)
display(tax)


,classification_level,full_classification,classification,max_pid,ambiguous,counts,domain,supergroup,division,class,order,family,genus,species
transcript_name,,,,,,,,,,,,,,
megahit_HN017_k141_199119.p1,family,Eukaryota; Eukaryota incertae sedis; Picozoa; ...,SG_Archaeplastida_Indeterminate,47.821,0,NaN,Eukaryota,Eukaryota incertae sedis,Picozoa,SG_Archaeplastida_Indeterminate,SG_Archaeplastida_Indeterminate,SG_Archaeplastida_Indeterminate,Unknown,Unknown
transabyss_HN028_.k50R4401869.p1,order,Eukaryota; Eukaryota incertae sedis; Picozoa; ...,SG_Archaeplastida_Indeterminate,47.821,0,NaN,Eukaryota,Eukaryota incertae sedis,Picozoa,SG_Archaeplastida_Indeterminate,SG_Archaeplastida_Indeterminate,Unknown,Unknown,Unknown
transabyss_HN026_.k20R26149338.p1,family,Eukaryota; Alveolata; Dinophyta; Dinophyceae; ...,Suessiales,47.821,0,NaN,Eukaryota,Alveolata,Dinophyta,Dinophyceae,Suessiales,Suessiales,Unknown,Unknown
velvet_HN002_NODE_626153_length_322_cov_2.055901.p1,family,Eukaryota; Hacrobia; Haptophyta; Prymnesiophyc...,Prymnesiales,47.821,1,NaN,Eukaryota,Hacrobia,Haptophyta,Prymnesiophyceae,Prymnesiales,Unknown,Unknown,Unknown
transabyss_HN006_.k20R29630776.p1,genus,Eukaryota; Hacrobia; Haptophyta; Prymnesiophyc...,Phaeocystis cordata,47.821,0,NaN,Eukaryota,Hacrobia,Haptophyta,Prymnesiophyceae,Phaeocystales,Phaeocystaceae,Phaeocystis cordata,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
trinity_HN017_TRINITY_DN1354_c0_g1_i1.p1,genus,Eukaryota; Archaeplastida; Chlorophyta; Mamiel...,Ostreococcus RCC809,3710.250,0,NaN,Eukaryota,Archaeplastida,Chlorophyta,Mamiellophyceae,Mamiellales,Bathycoccaceae,Ostreococcus RCC809,Unknown
trinity_HN026_TRINITY_DN163019_c0_g1_i1.p1,order,Eukaryota; Opisthokonta; Fungi; Ascomycota,Ascomycota,3765.540,1,NaN,Eukaryota,Opisthokonta,Fungi,Ascomycota,Unknown,Unknown,Unknown,Unknown
trinity_HN017_TRINITY_DN1354_c0_g1_i7.p1,genus,Eukaryota; Archaeplastida; Chlorophyta; Mamiel...,Ostreococcus RCC809,3842.170,0,NaN,Eukaryota,Archaeplastida,Chlorophyta,Mamiellophyceae,Mamiellales,Bathycoccaceae,Ostreococcus RCC809,Unknown


## Merge Taxonomy with metaproteomics

In [37]:
#Match up proteomics raw counts with taxonomy and make csv
#transpose protein data to match modified sample names
pp = p.T

merged_prot_tax = tax.join(p, how="inner") # Merge a (proteomic counts) with tax on index
merged_prot_tax.index.name = tax.index.name # restores the name of the index from tax
display(merged_prot_tax)

,classification_level,full_classification,classification,max_pid,ambiguous,counts,domain,supergroup,division,class,order,family,genus,species,3_UW5_02,43_UW91_02,24_UW73_51,31_UW114_02,18_UW109_51,63_UW104_02,6_UW9_02,37_UW81_02,53_UW98_02,56_UW100_51,41_UW89_02,35_UW77_02,49_UW96_02,33_UW77_51,24_UW73_51.1,19_UW70_02,61_UW103_02,13_UW28_02,55_UW99_02,38_UW85_51,40_UW89_51,60_UW103_51,54_UW99_51,48_UW96_51
transcript_name,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
trinity_HN006_TRINITY_DN474789_c0_g1_i1.p1,class,Eukaryota; Alveolata; Dinophyta; Dinophyceae,Dinophyceae,50.052,0,NaN,Eukaryota,Alveolata,Dinophyta,Dinophyceae,Unknown,Unknown,Unknown,Unknown,476052.0,813721.0,0.0,0.0,0.0,0.0,311599.0,796492.0,261233.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
trinity_HN020_TRINITY_DN400_c0_g1_i15.p1,order,Eukaryota; Hacrobia; Cryptophyta; SG_Cryptista...,SG_Cryptista_Indeterminate,50.828,0,NaN,Eukaryota,Hacrobia,Cryptophyta,SG_Cryptista_Indeterminate,SG_Cryptista_Indeterminate,Unknown,Unknown,Unknown,0.0,0.0,0.0,423386.0,366461.0,219372.0,99150.9,0.0,166669.0,0.0,0.0,0.0,210347.0,0.0,0.0,191981.0,303810.0,296212.0,263529.0,0.0,0.0,137350.0,251460.0,0.0
megahit_HN006_k141_405246.p2,division,Bacteria; Proteobacteria; Alphaproteobacteria,Alphaproteobacteria,51.216,0,NaN,Bacteria,Proteobacteria,Alphaproteobacteria,Unknown,Unknown,Unknown,Unknown,Unknown,158540.0,156984.0,0.0,0.0,0.0,0.0,304006.0,0.0,227671.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
trinity_HN020_TRINITY_DN198349_c0_g1_i1.p1,class,Bacteria; Proteobacteria; Gammaproteobacteria;...,Moraxellales,51.895,0,NaN,Bacteria,Proteobacteria,Gammaproteobacteria,Moraxellales,Unknown,Unknown,Unknown,Unknown,1530000.0,920190.0,0.0,0.0,0.0,322058.0,1020000.0,1150000.0,924322.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
trinity_HN002_TRINITY_DN459598_c0_g1_i1.p2,order,Eukaryota; Alveolata; Dinophyta; Dinophyceae; ...,Gymnodiniales,52.283,0,NaN,Eukaryota,Alveolata,Dinophyta,Dinophyceae,Gymnodiniales,Unknown,Unknown,Unknown,32656.7,0.0,2100000.0,0.0,0.0,0.0,62278.9,0.0,0.0,0.0,0.0,0.0,0.0,660970.0,2381050.0,0.0,74257.3,576652.0,353800.0,0.0,0.0,1030040.0,922950.0,23138000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
megahit_HN012_k141_79695.p1,genus,Eukaryota; Stramenopiles; Stramenopiles_X; Pel...,Pelagophyceae_XX,2987.600,1,NaN,Eukaryota,Stramenopiles,Stramenopiles_X,Pelagophyceae,Pelagophyceae_X,Pelagophyceae_XX,Unknown,Unknown,160180.0,362994.0,0.0,132319.0,0.0,0.0,94716.6,351280.0,274573.0,440935.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
megahit_HN006_k141_277541.p1,order,Eukaryota; Hacrobia; Haptophyta; Prymnesiophyceae,Prymnesiophyceae,3038.040,1,NaN,Eukaryota,Hacrobia,Haptophyta,Prymnesiophyceae,Unknown,Unknown,Unknown,Unknown,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3015710.0,3977500.0,2124750.0,5306100.0,3829560.0,2317760.0,3070240.0,2503710.0,2257990.0,3618350.0,2823000.0,3453900.0,3343190.0,3516680.0
megahit_HN017_k141_105730.p1,genus,Eukaryota; Archaeplastida; Chlorophyta; Mamiel...,Ostreococcus RCC809,3322.250,0,NaN,Eukaryota,Archaeplastida,Chlorophyta,Mamiellophyceae,Mamiellales,Bathycoccaceae,Ostreococcus RCC809,Unknown,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3959500.0,3732740.0,3802130.0,3008730.0,7110670.0,1003040.0,2581710.0,1221850.0,3365050.0,3237670.0,3265060.0,6052290.0,6761480.0,3849050.0


## Read functional annotations

In [38]:
# Read functional annotations file
egg = pd.read_csv("../data/metaP/transcriptome_db/eggnog.emapper.annotations", sep="\t", header=None, comment="#")

# Define column names
egg.columns = [
    "transcript_name", "seed_ortholog", "evalue", "score", "eggNOG_OG", "narr_OG_name",
    "narrOG_cat", "narr_OG_desc", "best_OG_names", "best_OG_cat", "best_OG_desc",
    "Preferred_name", "GOs", "EC", "KEGG_ko", "KEGG_pathway", "KEGG_Module",
    "KEGG_Reaction", "KEGG_rclass", "BRITE", "KEGG_TC", "CAZy", "BiGG_Reaction", "PFAMs"
]
#Set index
egg.set_index('transcript_name',inplace=True)
display(egg)

,seed_ortholog,evalue,score,eggNOG_OG,narr_OG_name,narrOG_cat,narr_OG_desc,best_OG_names,best_OG_cat,best_OG_desc,Preferred_name,GOs,EC,KEGG_ko,KEGG_pathway,KEGG_Module,KEGG_Reaction,KEGG_rclass,BRITE,KEGG_TC,CAZy,BiGG_Reaction,PFAMs
transcript_name,,,,,,,,,,,,,,,,,,,,,,,
megahit_HN001_k141_10.p1,745014.OMB55_00009280,6.400000e-90,337.8,"COG2812@1|root,COG2812@2|Bacteria,1MVCK@1224|P...",1J4KC@118884|unclassified Gammaproteobacteria,H,"DNA polymerase III is a complex, multichain en...",1RMIA@1236|Gammaproteobacteria,L,"DNA polymerase III is a complex, multichain en...",dnaX,"GO:0003674,GO:0003824,GO:0005488,GO:0005515,GO...",2.7.7.7,ko:K02343,"ko00230,ko00240,ko01100,ko03030,ko03430,ko0344...",M00260,"R00375,R00376,R00377,R00378",RC02795,"ko00000,ko00001,ko00002,ko01000,ko03032,ko03400",-,-,-,"DNA_pol3_delta2,DNA_pol3_gamma3,DNA_pol3_tau_4..."
megahit_HN001_k141_100008.p1,1400525.JNIU01000001_gene806,4.600000e-161,573.9,"COG0465@1|root,COG0465@2|Bacteria,1MU6J@1224|P...",4BPBS@82117|unclassified Alphaproteobacteria,O,"Acts as a processive, ATP-dependent zinc metal...",2TSYQ@28211|Alphaproteobacteria,O,"Acts as a processive, ATP-dependent zinc metal...",ftsH,-,-,ko:K03798,-,M00742,-,-,"ko00000,ko00002,ko01000,ko01002,ko03110",-,-,-,"AAA,FtsH_ext,Peptidase_M41"
megahit_HN001_k141_100035.p1,2903.EOD28543,4.100000e-61,241.1,"COG2072@1|root,KOG1399@2759|Eukaryota",KOG1399@2759|Eukaryota,P,"N,N-dimethylaniline monooxygenase activity",KOG1399@2759|Eukaryota,P,"N,N-dimethylaniline monooxygenase activity",-,-,1.14.13.8,ko:K00485,"ko00982,ko01120,map00982,map01120",-,R08266,RC02233,"ko00000,ko00001,ko01000",-,-,-,"FMO-like,K_oxygenase"
megahit_HN001_k141_100042.p1,1397666.RS24_01239,1.900000e-39,168.7,"COG1298@1|root,COG1298@2|Bacteria,1MUF3@1224|P...",4BPKT@82117|unclassified Alphaproteobacteria,N,Required for formation of the rod structure of...,2TQUR@28211|Alphaproteobacteria,N,Required for formation of the rod structure of...,flhA,-,-,ko:K02400,"ko02040,map02040",-,-,-,"ko00000,ko00001,ko02035,ko02044","3.A.6.2,3.A.6.3",-,-,FHIPEP
megahit_HN001_k141_100044.p2,2903.EOD13626,1.000000e-132,479.9,"COG5245@1|root,KOG3595@2759|Eukaryota",KOG3595@2759|Eukaryota,Z,dynein light chain binding,KOG3595@2759|Eukaryota,Z,dynein light chain binding,-,-,-,ko:K10408,"ko05016,map05016",-,-,-,"ko00000,ko00001,ko04812",-,-,-,"AAA_5,AAA_6,AAA_7,AAA_8,AAA_9,DHC_N1,DHC_N2,Dy..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
velvet_HN044_NODE_99566_length_498_cov_5.244980.p1,755732.Fluta_1747,4.600000e-73,280.8,"COG0495@1|root,COG0495@2|Bacteria,4NE5K@976|Ba...",2PAA7@246874|Cryomorphaceae,J,"PFAM tRNA synthetases class I (I, L, M and V)",4NE5K@976|Bacteroidetes,J,Belongs to the class-I aminoacyl-tRNA syntheta...,leuS,"GO:0003674,GO:0003824,GO:0004812,GO:0004823,GO...",6.1.1.4,ko:K01869,"ko00970,map00970","M00359,M00360",R03657,"RC00055,RC00523","ko00000,ko00001,ko00002,ko01000,ko01007,ko0301...",-,-,-,"Anticodon_1,DUF559,tRNA-synt_1,tRNA-synt_1_2"
velvet_HN044_NODE_99652_length_1177_cov_2.969414.p1,3218.PP1S70_155V6.1,2.900000e-11,76.6,"28J5E@1|root,2QRHK@2759|Eukaryota,37JZE@33090|...",3G96C@35493|Streptophyta,-,-,3G96C@35493|Streptophyta,-,-,-,-,-,-,-,-,-,-,-,-,-,-,Lung_7-TM_R
velvet_HN044_NODE_99715_length_704_cov_3.177557.p1,44056.XP_009033785.1,5.800000e-34,151.4,"KOG0674@1|root,KOG0674@2759|Eukaryota",KOG0674@2759|Eukaryota,O,unfolded protein binding,KOG0674@2759|Eukaryota,O,unfolded protein binding,CALR,"GO:0000003,GO:0000122,GO:0001669,GO:0001846,GO...",-,"ko:K08057,ko:K10098","ko04141,ko04145,ko04612,ko05142,ko05166,map041...",-,-,-,"ko00000,ko00001,ko03110,ko04091",-,-,-,"ANF_receptor,CTD_bind,Calreticulin,EF-hand_4,F..."


## Merge Meta proteomics with taxonomy and functional annotations and print csv file

In [39]:
# Merge functional annotations with merged prot,tax dataframe based on transcript_name
merged_prot_tax_ann = egg.join(merged_prot_tax, how="inner")
merged_prot_tax_ann.index.name = egg.index.name # restores the name of the index from egg
#display(merged_prot_tax_ann)

# Sort proteomics results based on max values from results (column 30 on)
merged_prot_tax_ann = merged_prot_tax_ann.assign(
    max_val=merged_prot_tax_ann.iloc[:, 38:].max(axis=1)
).sort_values(by='max_val', ascending=False).drop(columns='max_val')
display(merged_prot_tax_ann)

#Create a CSV file with proteomics, taxonomy and functional annotations together in the same file
# Create folder if it doesn't exist
os.makedirs("merged_files", exist_ok=True)
#create file
merged_prot_tax_ann.to_csv(f"../data/metaP/merged_prot_tax_ann_{sample_set}.csv", index=True)

,seed_ortholog,evalue,score,eggNOG_OG,narr_OG_name,narrOG_cat,narr_OG_desc,best_OG_names,best_OG_cat,best_OG_desc,Preferred_name,GOs,EC,KEGG_ko,KEGG_pathway,KEGG_Module,KEGG_Reaction,KEGG_rclass,BRITE,KEGG_TC,CAZy,BiGG_Reaction,PFAMs,classification_level,full_classification,classification,max_pid,ambiguous,counts,domain,supergroup,division,class,order,family,genus,species,3_UW5_02,43_UW91_02,24_UW73_51,31_UW114_02,18_UW109_51,63_UW104_02,6_UW9_02,37_UW81_02,53_UW98_02,56_UW100_51,41_UW89_02,35_UW77_02,49_UW96_02,33_UW77_51,24_UW73_51.1,19_UW70_02,61_UW103_02,13_UW28_02,55_UW99_02,38_UW85_51,40_UW89_51,60_UW103_51,54_UW99_51,48_UW96_51
transcript_name,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
megahit_HN002_k141_143656.p1,51511.ENSCSAVP00000014853,6.000000e-62,243.4,"COG5277@1|root,KOG0676@2759|Eukaryota,38C0D@33...",485B3@7711|Chordata,Z,mesenchyme migration,3BBGP@33208|Metazoa,Z,ATP binding,ACTG2,"GO:0000166,GO:0000287,GO:0001568,GO:0001655,GO...",-,"ko:K05692,ko:K10354,ko:K12313,ko:K12314,ko:K12315","ko04015,ko04145,ko04210,ko04260,ko04261,ko0427...",-,-,-,"ko00000,ko00001,ko03019,ko03021,ko03029,ko0303...",-,-,-,Actin,NaN,NaN,NaN,232.80,1,NaN,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,1130000.0,694616.0,9330000.0,300802.0,2.910000e+06,1010000.0,3410000.0,736246.0,1800000.0,2650000.0,44965000.0,61484600.0,53557100.0,3.726180e+09,291791000.0,47452100.0,48706200.0,279375000.0,69728200.0,4.341600e+08,283575000.0,293864000.0,402608000.0,2.157200e+10
transabyss_HN032_.k50S4540205.p1,44689.DDB0220457,8.500000e-137,493.0,"COG5277@1|root,KOG0676@2759|Eukaryota,3X8DK@55...",3X8DK@554915|Amoebozoa,Z,Actins are highly conserved proteins that are ...,3X8DK@554915|Amoebozoa,Z,Actins are highly conserved proteins that are ...,-,"GO:0000278,GO:0000281,GO:0000323,GO:0000902,GO...",-,ko:K10355,-,-,-,-,"ko00000,ko04812",-,-,-,Actin,supergroup,Eukaryota,Eukaryota,494.70,1,NaN,Eukaryota,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,9901530.0,17077400.0,7209760.0,1.854990e+09,89323700.0,9805550.0,5097540.0,38635000.0,3106440.0,1.079600e+09,601316000.0,347568000.0,566255000.0,1.703460e+09
trinity_HN021_TRINITY_DN51095_c0_g1_i1.p1,159749.E7BWH1,1.300000e-45,188.7,"28JG1@1|root,2QRV6@2759|Eukaryota,2XEM0@2836|B...",2XEM0@2836|Bacillariophyta,P,One of the components of the core complex of p...,2XEM0@2836|Bacillariophyta,P,One of the components of the core complex of p...,psbB,-,-,ko:K02704,"ko00195,ko01100,map00195,map01100",M00161,-,-,"ko00000,ko00001,ko00002,ko00194",-,-,-,PSII,supergroup,Eukaryota,Eukaryota,183.33,1,NaN,Eukaryota,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,598962.0,10200000.0,11500000.0,139000000.0,1.180000e+09,27800000.0,438560.0,6010000.0,11000000.0,151000000.0,1280370.0,1368230.0,1400960.0,1.588680e+07,10635700.0,808854.0,65684000.0,583152.0,129531000.0,4.448580e+06,5213880.0,112518000.0,218175000.0,2.838560e+06
trinity_HN017_TRINITY_DN64792_c0_g1_i3.p1,103372.F4WEV7,1.700000e-60,238.4,"COG5277@1|root,KOG0676@2759|Eukaryota,38C0D@33...",46JEY@7399|Hymenoptera,Z,Actin,3BBGP@33208|Metazoa,Z,ATP binding,ACTB,"GO:0000003,GO:0000228,GO:0000278,GO:0000281,GO...",-,ko:K05692,"ko04015,ko04145,ko04210,ko04390,ko04391,ko0451...",-,-,-,"ko00000,ko00001,ko03019,ko03021,ko03029,ko0303...",-,-,-,Actin,NaN,NaN,NaN,232.80,1,NaN,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,5956910.0,28368800.0,6312380.0,1.084160e+09,143669000.0,6946920.0,1420780.0,24319700.0,2917580.0,5.338550e+08,240077000.0,268530000.0,140549000.0,3.625280e+08
transabyss_HN016_.k50R5006130.p2,8010.XP_010892483.1,2.200000e-221,774.6,"COG5277@1|root,KOG0676@2759|Eukaryota,38C0D@33...",49XXA@7898|Actinopterygii,Z,"Actin, alpha",3BBGP@33208|Metazoa,Z,ATP binding,-,"GO:0005575,GO:0005622,GO:0005623,GO:0005856,GO...",-,"ko:K05692,ko:K10354,ko:K12313,ko:K12314","ko04015,ko04145,ko04210,ko04260,ko04261,ko042

## STOP HERE

## Do I need to sort the data based on protein abundance at this point?

In [ ]:
#Sort values based on first result's abundance - other samples usually have similar patterns
#Change the name of the column below based on the desired column to sort results by

merged_prot_tax_ann_sorted = merged_prot_tax_ann.sort_values(by=merged_prot_tax_ann.columns[33])

#merged_prot_tax_ann = merged_prot_tax_ann.sort_values(by="161111_BATS_2D_313_0m",ascending=False) #Sort by the first proteome result from highest to lowest
#display(merged_prot_tax_ann)

# Show first rows with ALL headers visible
display(merged_prot_tax_ann_sorted.head())

,seed_ortholog,evalue,score,eggNOG_OG,narr_OG_name,narrOG_cat,narr_OG_desc,best_OG_names,best_OG_cat,best_OG_desc,Preferred_name,GOs,EC,KEGG_ko,KAGG_pathway,KEGG_Module,KEGG_Reaction,KEGG_rclass,BRITE,KEGG_TC,CAZy,BiGG_Reaction,PFAMs,classification_level,full_classification,classification,max_pid,ambiguous,counts,domain,supergroup,division,class,order,family,genus,species,NRFBMC1457PR1A,NRFBMC1454PR1B,NRNEMC1457PR2B,NRNEMC1445PR1B,NRFBS10PR2A,NRNEMC1449PR2B,NRNES11PR1A,NRFBS10PR1A,NRFBMC1445PR1B,NRFBMC1454PR2B,NRFBMC1449PR1B,NRNEMC1457PR1A,NRFBMC1454PR1A,NRNES10PR2B,NRFBMC1457PR2A,NRFBMC1449PR2B,NRNEMC1445PR2B,NRNEMC1449PR1A,NRNEMC1454PR1A,NRFBMC1454PR2A,NRNES11PR1B,NRNEMC1445PR2A,NRNEMC1454PR1B,NRNEMC1449PR2A,NRNES10PR2A,NRNES11PR2B,NRNEMC1457PR1B,NRFBMC1457PR2B,NRFBMC1457PR1B,NRNEMC1454PR2A,NRNEMC1457PR2A,NRFBS11PR1A,NRFBMC1445PR2B,NRNES10PR1A,NRFBS10PR1B,NRFBS11PR2A,NRNEMC1454PR2B,NRFBMC1449PR2A,NRNES10PR1B,NRNEMC1449PR1B,NRNES11PR2A,NRFBS10PR2B,NRFBMC1445PR2A,NRFBMC1449PR1A,NRNEMC1445PR1A
megahit_HN029_k141_15164.p2,5827.XP_004220568.1,7.000000e-57,226.9,"COG5126@1|root,KOG0028@2759|Eukaryota,3YB6J@57...",3YXR7@5819|Haemosporida,DZ,EF-hand domain,3KAMF@422676|Aconoidasida,DZ,EF-hand domain,-,-,-,ko:K16465,-,-,-,-,"ko00000,ko03036",-,-,-,"EF-hand_1,EF-hand_5,EF-hand_7,EF-hand_8",genus,Eukaryota; Alveolata; Dinophyta; Dinophyceae; ...,Abedinium,265.78,0,NaN,Eukaryota,Alveolata,Dinophyta,Dinophyceae,Abediniales,SG_Alveolata_Indeterminate,Abedinium,NaN,0.0,0.0,145584.0,0.0,0.0,0.0,212047.0,0.0,0.0,0.0,0.0,138594.0,0.0,0.0,0.0,0.0,0.0,17546.1,40793.4,0.0,87487.9,0.0,35507.5,18921.1,0.0,0.0,119835.0,0.0,0.0,43400.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,97033.1,0.0,0.0,0.0,0.0
megahit_HN005_k141_100059.p1,406327.Mevan_1556,4.300000e-145,521.2,"COG0334@1|root,arCOG01352@2157|Archaea,2XU0F@2...",2XU0F@28890|Euryarchaeota,E,Belongs to the Glu Leu Phe Val dehydrogenases ...,2XU0F@28890|Euryarchaeota,E,Belongs to the Glu Leu Phe Val dehydrogenases ...,gdhA,"GO:0005575,GO:0005622,GO:0005623,GO:0005737,GO...","1.4.1.2,1.4.1.3,1.4.1.4","ko:K00260,ko:K00261,ko:K00262","ko00220,ko00250,ko00430,ko00471,ko00910,ko0110...",M00740,"R00243,R00248","RC00006,RC02799","ko00000,ko00001,ko00002,ko01000,ko04147",-,-,-,"ELFV_dehydrog,ELFV_dehydrog_N",family,Eukaryota; Alveolata; Dinophyta; Dinophyceae; ...,SG_Alveolata_Indeterminate,594.61,0,NaN,Eukaryota,Alveolata,Dinophyta,Dinophyceae,Abediniales,SG_Alveolata_Indeterminate,None,NaN,29349.6,66272.7,59917.6,0.0,110566.0,0.0,0.0,113174.0,0.0,50892.3,0.0,51706.3,0.0,0.0,32247.8,41746.3,0.0,44644.0,104373.0,55613.0,90934.6,60386.7,111169.0,39959.0,0.0,89043.8,64736.0,0.0,31542.0,68853.3,144622.0,45030.9,0.0,189185.0,129321.0,0.0,0.0,0.0,202406.0,0.0,87422.7,103482.0,0.0,36617.1,0.0
trinity_HN037_TRINITY_DN2359_c1_g1_i1.p1,38833.XP_003062207.1,8.000000e-71,273.9,"COG0652@1|root,KOG0865@2759|Eukaryota,37M44@33...",34JBX@3041|Chlorophyta,O,PPIases accelerate the folding of proteins. It...,34JBX@3041|Chlorophyta,O,PPIases accelerate the folding of proteins. It...,-,-,5.2.1.8,"ko:K01802,ko:K03767,ko:K09565","ko01503,ko04020,ko04022,ko04217,ko05012,ko0501...",-,-,-,"ko00000,ko00001,ko01000,ko03110,ko04147",-,-,-,Pro_isomerase,family,Eukaryota; Alveolata; Dinophyta; Dinophyceae; ...,SG_Alveolata_Indeterminate,343.38,0,NaN,Eukaryota,Alveolata,Dinophyta,Dinophyceae,Abediniales,SG_Alveolata_Indeterminate,None,NaN,57857.5,0.0,406626.0,482792.0,213207.0,690385.0,0.0,0.0,140242.0,92221.7,134128.0,307414.0,0.0,624889.0,90319.8,227959.0,132324.0,436391.0,362785.0,82724.6,394706.0,491315.0,292723.0,587325.0,642011.0,578236.0,314130.0,0.0,72541.9,424404.0,403801.0,110894.0,86461.1,753576.0,32051.5,0.0,0.0,71986.3,561244.0,711936.0,375043.0,154637.0,80916.8,124563.0,145769.0
trinity_HN041_TRINITY_DN143063_c0_g1_i1.p1,1112217.PPL19_10867,4.300000e-80,305.1,"COG0372@1|root,COG0372@2|Bacteria,1MUKX@1224|P...",1RNT1@1236|Gammaproteobacteria,C,Belongs to the citrate synthase family,1RNT1@1236|Gammaproteobacteria,C,Belongs to

In [ ]:
### Read metadata
# meta = pd.read_csv(f"Metadata_{cruises}.csv", skiprows=1)
# meta.index = meta["Sample_ID"]
## Removes spaces from names
# meta.columns = meta.columns.str.replace(" ", "_")


In [ ]:
# Convert to datetime format, setting errors='coerce' to handle invalid formats
meta["Cast_date_GMT_(dd-mmm-yyyy)"] = pd.to_datetime(meta["Cast_date_GMT_(dd-mmm-yyyy)"], format='mixed', errors='coerce')

#Convert to decimal degrees
def date_to_decimal(date):
    if pd.isna(date):  # Handle NaT values
        return None
    year = date.year
    doy = date.timetuple().tm_yday  # Day of the year
    total_days = 366 if (year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)) else 365  # Leap year check
    return year + (doy - 1) / total_days

# Insert Decimal_Date column right next to Cast_date_GMT_(dd-mmm-yyyy)
meta.insert(10, "Decimal_Date", meta["Cast_date_GMT_(dd-mmm-yyyy)"].apply(date_to_decimal))
meta.rename(columns={"Cast_date_GMT_(dd-mmm-yyyy)": "Cast_date_GMT"}, inplace=True)

#display(meta)


In [ ]:
#Create a new variable to transpose prot,tax,func anno merged data file to be able to connect with metadata
merged_all_ann = merged_prot_tax_ann.copy()
#Transpose merged prot,tax,ann file
merged_transposed = merged_all_ann.T
#display(merged_transposed)

In [ ]:
#merge prot,tax,ann with metadata and print a merged document including all data in csv format
meta_merged_prot_tax_ann = meta.merge(merged_transposed, left_index=True, right_index=True, how="outer") #use "inner" to only copy exact matches such as only sample ID
display(meta_merged_prot_tax_ann)

import os

# Create folder if it doesn't exist
os.makedirs("merged_files", exist_ok=True)

meta_merged_prot_tax_ann.to_csv(f"merged_files/meta_merged_prot_tax_ann_{cruises}.csv", index=True)

,Cruise_ID,Sample_type,Depth,Time_on_the_day,Unique_ID,Station,Lat_(dec_deg),Lon_(dec_deg),Sampling_method,Cast_date_GMT,...,transabyss_HN002_.k50R5604404.p1,transabyss_HN002_.k50R5608511.p1,transabyss_HN002_.k50R5613023.p1,transabyss_HN002_.k50R5620945.p1,transabyss_HN002_.k50R5621724.p1,transabyss_HN002_.k50R5622472.p1,transabyss_HN002_.k50R5626464.p1,transabyss_HN002_.k50R5627471.p1,transabyss_HN002_.k50R5628714.p1,velvet_HN044_NODE_95149_length_383_cov_5.396867.p1
Sample_ID,,,,,,,,,,,,,,,,,,,,,
160916_2D_BATS_325_30m,10325.0,30M,NaN,NaN,NaN,BATS,31,-64,McLane pump,2016-06-16,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
160916_2D_BATS_325_chl,10325.0,DCM,NaN,NaN,NaN,BATS,31,-64,McLane pump,2016-06-16,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
160916_2D_BATS_325_inline_AM,10325.0,UWAM,NaN,NaN,NaN,BATS,31.666,-64.178,Underway system,2016-06-16,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
160916_2D_BATS_325_inline_PM,10325.0,UWPM,NaN,NaN,NaN,BATS,31.666,-64.167,Underway system,2016-06-16,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
160916_2D_BATS_326_30m,10326.0,30M,NaN,NaN,NaN,BATS,31,-64,McLane pump,2016-07-23,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
NaN,20358.0,UWAM,5.0,day,203581.0,NaN,NaN,NaN,NaN,2019-04-21,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NaN,20358.0,UWPM,5.0,night,203582.0,NaN,NaN,NaN,NaN,2019-04-22,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NaN,20358.0,DCM,85.0,day,203584.0,NaN,NaN,NaN,NaN,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
#merge prot with metadata and print a merged document to be used for figures and data analysis
meta_merged_prot = meta.merge(merged_transposed, left_index=True, right_index=True, how="inner") #use "outer" to merge all data in both documents and not only exact matches
display(meta_merged_prot)

import os

# Create folder if it doesn't exist
os.makedirs("merged_files", exist_ok=True)

meta_merged_prot.to_csv(f"merged_files/meta_merged_prot_{cruises}.csv", index=True)

,Cruise_ID,Sample_type,Depth,Time_on_the_day,Unique_ID,Station,Lat_(dec_deg),Lon_(dec_deg),Sampling_method,Cast_date_GMT,...,transabyss_HN002_.k50R5604404.p1,transabyss_HN002_.k50R5608511.p1,transabyss_HN002_.k50R5613023.p1,transabyss_HN002_.k50R5620945.p1,transabyss_HN002_.k50R5621724.p1,transabyss_HN002_.k50R5622472.p1,transabyss_HN002_.k50R5626464.p1,transabyss_HN002_.k50R5627471.p1,transabyss_HN002_.k50R5628714.p1,velvet_HN044_NODE_95149_length_383_cov_5.396867.p1
Sample_ID,,,,,,,,,,,,,,,,,,,,,
161111_BATS_2D_313_0m,10313,UWAM,NaN,NaN,NaN,BATS,31.666,-64.167,NaN,2015-04-07,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
161122_BATS_2D_313_PM,10313,UWPM,NaN,NaN,NaN,BATS,31.666,-64.167,NaN,2015-04-07,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
161111_BATS_2D_313_30m,10313,30M,NaN,NaN,NaN,BATS,31.666,-64.167,NaN,2015-04-07,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
161111_BATS_2D_313_82m,10313,82M,NaN,NaN,NaN,BATS,31.666,-64.167,NaN,2015-04-07,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
161111_BATS_2D_313_90m,10313,90M,NaN,NaN,NaN,BATS,31.666,-64.167,NaN,2015-04-07,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
181018_2D_BATS_334dcm,10334,DCM,NaN,NaN,NaN,BATS,31,-64,McLane pump,2017-04-08,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
181018_2D_BATS_335am,10335,UWAM,NaN,NaN,NaN,BATS,31.666,-64.167,Underway system,2017-05-09,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
181018_2D_BATS_335pm,10335,UWPM,NaN,NaN,NaN,BATS,31.672,-64.161,Underway system,2017-05-10,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
